# 📘 Agentic 架构 18：Mem0 长期记忆

欢迎来到本系列第 18 个 Notebook。这里我们聚焦一个非常实用的能力：**让智能体拥有可持续积累、可按需检索的长期记忆**。在许多真实应用里，真正的差异化并不只是模型会不会回答，而是它能否记住用户是谁、偏好什么、过去做过什么，以及这些信息怎样在下一次交互里继续发挥作用。

`Mem0` 就是专门为这类场景设计的记忆层。它帮助我们把“对话历史”升级成“结构化、可搜索、可复用的长期记忆”。

### 定义
**Mem0 长期记忆架构**指的是：把用户与智能体交互中的关键事实、偏好、约束、目标等信息，抽取并保存为可检索的记忆条目；在后续对话中，智能体先召回相关记忆，再基于这些记忆生成更加个性化、连续一致的回答。

### 高层工作流
1. **接收消息：** 用户与智能体发生一次真实交互。
2. **写入记忆：** 将消息对提交给 `Mem0`，由其抽取值得长期保留的信息。
3. **检索记忆：** 下一次提问时，根据查询语义召回相关记忆。
4. **增强回答：** 把召回的记忆注入提示词，生成更加个性化的回复。
5. **持续更新：** 随着交互增加，记忆会不断累积、更新和清理。

### 适用场景 / 应用
- **长期个人助手：** 记住用户偏好、项目、风格与限制条件。
- **客服 / 销售 Agent：** 记住客户历史需求，提高连续服务质量。
- **学习辅导系统：** 记住学习目标、薄弱点、进度和反馈。
- **多轮业务自动化：** 在长周期流程中保持上下文一致性。

### 优势与局限
- **优势：**
  - 让智能体具备跨会话的连续性，而不只依赖单次上下文窗口。
  - 检索的是“真正重要的用户事实”，而不是无差别堆叠全部历史消息。
- **局限：**
  - 记忆质量依赖抽取策略，错误记忆也可能被保存。
  - 涉及用户数据时，必须认真考虑隐私、权限与删除机制。


## 阶段 0：环境准备

本 Notebook 使用 `Mem0 Platform` 的 `MemoryClient` 演示最核心的记忆流程：`add`、`search`、`get_all`、`delete_all`。

**运行前需要准备的环境变量：**
```bash
MEM0_API_KEY="your_mem0_api_key"
DEEPSEEK_API_KEY="your_deepseek_api_key"  # 可选，用于最后的记忆增强回复示例
```

如果你暂时没有 `DEEPSEEK_API_KEY`，也没关系，前面的记忆写入与检索流程仍然可以独立运行。


In [ ]:
# !pip install -q -U mem0ai openai python-dotenv rich


In [2]:
import os
import json
from dotenv import load_dotenv
from mem0 import MemoryClient
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel

load_dotenv(override=True)
console = Console()

mem0_api_key = os.getenv("MEM0_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

if not mem0_api_key:
    raise ValueError("未检测到 MEM0_API_KEY，请先在 .env 中配置该变量。")

memory_client = MemoryClient(api_key=mem0_api_key)

llm_client = None
if deepseek_api_key:
    llm_client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")

USER_ID = "mem0-demo-zh"

def dump_json(data):
    print(json.dumps(data, ensure_ascii=False, indent=2, default=str))

console.print(Panel("Mem0 客户端已初始化。", title="环境就绪", border_style="green"))
if not llm_client:
    console.print("[yellow]未检测到 DEEPSEEK_API_KEY，最后一节将仅展示检索结果，不调用 LLM。[/yellow]")


╭─────────────────────────────────────────────────── 环境就绪 ────────────────────────────────────────────────────╮
│ Mem0 客户端已初始化。                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 阶段 1：写入用户长期记忆

这里我们模拟三轮对话，把用户的偏好、学习目标和饮食限制写入 `Mem0`。注意我们不是手工维护字典，而是把真实消息对交给 `Mem0` 做记忆抽取，这也是它在实际系统中的关键价值。


In [3]:
sample_conversations = [
    [
        {"role": "user", "content": "你好，我叫小林，我正在系统学习 AI Agent，希望每周末都能做项目练习。"},
        {"role": "assistant", "content": "收到，我会记住你正在学习 AI Agent，而且偏好周末做项目练习。"}
    ],
    [
        {"role": "user", "content": "我比较喜欢中文讲解，最好配合代码示例，不喜欢太抽象的理论。"},
        {"role": "assistant", "content": "明白，我会优先用中文、配代码示例，并尽量减少抽象表述。"}
    ],
    [
        {"role": "user", "content": "另外我午饭通常吃清淡一点，而且对花生过敏。"},
        {"role": "assistant", "content": "好的，我会记住你偏好清淡饮食，并避免推荐含花生的食物。"}
    ]
]

for idx, messages in enumerate(sample_conversations, start=1):
    result = memory_client.add(messages, user_id=USER_ID)
    console.print(f"[green]第 {idx} 组记忆已写入。[/green]")
    dump_json(result)


第 1 组记忆已写入。

{
  "results": [
    {
      "message": "Memory processing has been queued for background execution",
      "status": "PENDING",
      "event_id": "5fac2db0-5aa6-4e55-ac80-d2650b733f9d"
    }
  ]
}


第 2 组记忆已写入。

{
  "results": [
    {
      "message": "Memory processing has been queued for background execution",
      "status": "PENDING",
      "event_id": "de4c9f5b-0b84-4919-a4ec-7c778719bbe3"
    }
  ]
}


第 3 组记忆已写入。

{
  "results": [
    {
      "message": "Memory processing has been queued for background execution",
      "status": "PENDING",
      "event_id": "3fbb0cbf-a42b-4c01-b281-dcf63f58a8c7"
    }
  ]
}


## 阶段 2：检索与查看记忆

在真实产品里，智能体不会把所有历史消息都塞进提示词，而是先按问题检索最相关的记忆。下面分别演示：

- `search`：按当前问题召回最相关记忆
- `get_all`：查看某个用户当前保存的全部记忆


In [4]:
query = "这个用户喜欢什么学习方式？饮食上有什么需要注意的地方？"

search_result = memory_client.search(query, filters={"user_id": USER_ID})
all_memories = memory_client.get_all(filters={"user_id": USER_ID})

console.print(Panel(query, title="检索问题", border_style="cyan"))
print("===== search() 返回结果 =====")
dump_json(search_result)

print("\n===== get_all() 返回结果 =====")
dump_json(all_memories)


╭─────────────────────────────────────────────────── 检索问题 ────────────────────────────────────────────────────╮
│ 这个用户喜欢什么学习方式？饮食上有什么需要注意的地方？                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

===== search() 返回结果 =====
{
  "results": [
    {
      "id": "096a7762-37f0-41eb-bf81-d037dc33df66",
      "memory": "User usually eats light lunch and is allergic to peanuts",
      "user_id": "mem0-demo-zh",
      "metadata": null,
      "categories": [
        "food",
        "health",
        "user_preferences"
      ],
      "created_at": "2026-03-31T03:34:27-07:00",
      "updated_at": "2026-03-31T03:35:29-07:00",
      "expiration_date": null,
      "structured_attributes": {
        "year": 2026,
        "month": 3,
        "day": 31,
        "hour": 10,
        "minute": 34,
        "day_of_week": "tuesday",
        "week_of_year": 14,
        "day_of_year": 90,
        "quarter": 1,
        "is_weekend": false
      },
      "score": 0.9
    },
    {
      "id": "3c7053c6-64ce-4134-8995-a6acfe1eee86",
      "memory": "User prefers Chinese explanations with code examples and dislikes abstract theory.",
      "user_id": "mem0-demo-zh",
      "metadata": null,
      "categories"

## 阶段 3：基于记忆增强回答

这一段演示最关键的产品化动作：**先检索记忆，再把检索结果作为额外上下文交给模型生成回答**。这也是让 Agent 呈现“你还记得我”的核心机制。


In [5]:
user_question = "请你给我安排一个周末的 AI Agent 学习计划，并顺便推荐一个适合我的午餐。"

memory_hits = memory_client.search(user_question, filters={"user_id": USER_ID})
memory_lines = []
for item in memory_hits.get("results", []):
    if isinstance(item, dict):
        memory_lines.append(f"- {item.get('memory', str(item))}")
    else:
        memory_lines.append(f"- {item}")

memory_context = "\n".join(memory_lines) if memory_lines else "- 暂未检索到相关记忆"

console.print(Panel(memory_context, title="召回的记忆", border_style="magenta"))

if llm_client:
    messages = [
        {
            "role": "system",
            "content": "你是一名带长期记忆的中文学习助手。请优先利用给定记忆，让建议更贴合用户偏好；不要编造记忆中没有的信息。"
        },
        {
            "role": "user",
            "content": f"用户问题：{user_question}\n\n可用记忆：\n{memory_context}\n\n请给出具体建议。"
        }
    ]

    response = llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=0.4
    )
    print(response.choices[0].message.content)
else:
    print("未配置 DEEPSEEK_API_KEY，因此本节只展示召回的记忆。你可以把这些记忆拼进任意 LLM 的提示词中，用于生成个性化回答。")


╭────────────────────────────────────────────────── 召回的记忆 ───────────────────────────────────────────────────╮
│ - 小林 is studying AI Agent systemically and prefers to do project practice every weekend.                      │
│ - User prefers Chinese explanations with code examples and dislikes abstract theory.                            │
│ - User usually eats light lunch and is allergic to peanuts                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

根据你的学习习惯和偏好，我为你安排了以下 **周末 AI Agent 学习计划**，并搭配了适合的午餐建议：

---

### **周六：基础实践 + 小项目启动**
**上午（9:00-12:00）**  
- **主题**：用 Python 实现一个简单的 Rule-Based Agent（基于规则的智能体）  
- **内容**：  
  1. 用 `if-else` 或有限状态机实现一个自动回复天气查询的 Agent  
  2. 代码示例（中文注释）：  
  ```python
  # 规则库：城市对应天气
  weather_rules = {
      "北京": "晴天",
      "上海": "多云",
      "广州": "小雨"
  }
  
  def rule_based_agent(city):
      if city in weather_rules:
          return f"{city}的天气是：{weather_rules[city]}"
      else:
          return "暂未收录该城市天气"
  
  # 测试
  print(rule_based_agent("北京"))  # 输出：北京的天气是：晴天
  ```

**下午（14:00-17:00）**  
- **主题**：为 Agent 添加工具调用能力（如调用 API）  
- 实践：用 `requests` 库接入免费天气 API，让 Agent 动态获取数据  
- 目标：完成一个可交互的命令行天气查询 Agent

---

### **周日：进阶整合 + 项目优化**
**上午（9:00-12:00）**  
- **主题**：将 Rule-Based Agent 升级为基于 LLM 的 Agent  
- 内容：  
  1. 使用 OpenAI API 或国内平台（如智谱、DeepSeek）的聊天接口  
  2. 示例：用 `openai` 库让 Agent 理解用户意图并调用天气工具  
  ```python
  # 伪代码示例：结合 LLM 判断用户是否需要天气查询
  def llm_agent(user_input):
      if "天气" in user_input:
      

## 阶段 4：清理演示数据

做实验时，经常需要清理某个测试用户的全部记忆。`Mem0` 提供了 `delete_all`，但为了防止误删，它要求你至少传入一个过滤条件。


In [ ]:
# 如需清理本 Notebook 写入的测试记忆，请取消下一行注释。
# cleanup_result = memory_client.delete_all(user_id=USER_ID)
# dump_json(cleanup_result)


## 结论

在这个 Notebook 中，我们用 `Mem0` 演示了一个长期记忆层的最小可用闭环：

1. 把多轮对话写入长期记忆。
2. 根据新问题检索相关记忆。
3. 把记忆作为上下文增强后续回答。

这类模式非常适合构建真正“越用越懂你”的 Agent。后续如果你想进一步扩展，可以继续加入：多用户隔离、记忆过期策略、显式更新记忆、隐私审计，以及和 LangGraph 工作流的深度集成。
